<a href="https://colab.research.google.com/github/alexdiehl0/Backtesting-Strategies_Crypto/blob/Main/ModernPortfolioTheory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pycoingecko PyPortfolioOpt pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 9.1 MB/s eta 0:00:00


In [ ]:
from pycoingecko import CoinGeckoAPI
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

cg = CoinGeckoAPI()

# --- 1. Handpicked Top 25 by Market Cap (CoinMarketCap snapshot, Nov 2025) ---
coin_ids = [
    "bitcoin",          # BTC
    "ethereum",         # ETH
    "ripple",           # XRP
    "bnb",              # BNB
    "solana",           # SOL
    "tron",             # TRX
    "dogecoin",         # DOGE
    "cardano",          # ADA
    "hyperliquid",      # HYPE
    "chainlink",        # LINK
    "bitcoin-cash",     # BCH
    "stellar",          # XLM
    "unus-sed-leo",     # LEO
    "sui",              # SUI
    "hedera",           # HBAR
    "avalanche-2",      # AVAX
    "litecoin",         # LTC
    "monero",           # XMR
    "zcash",            # ZEC
    "shiba-inu",        # SHIB
    "the-open-network", # TON
]

# Excluded stablecoins (not suitable for MPT)
excluded = {"tether", "usd-coin", "dai", "ethena-usde"}

print(f"Fetching daily prices for {len(coin_ids)} major non-stable cryptocurrencies:")
print(coin_ids)

# --- 2. Define fetch function ---
def fetch_coin(coin, days=365):
    try:
        data = cg.get_coin_market_chart_by_id(id=coin, vs_currency="usd", days=days)
        prices = pd.DataFrame(data["prices"], columns=["timestamp", "price"])
        prices["date"] = pd.to_datetime(prices["timestamp"], unit="ms")
        prices = prices.set_index("date")["price"]
        print(f"✅ {coin} fetched successfully.")
        return coin, prices
    except Exception as e:
        print(f"⚠️ Error fetching {coin}: {e}")
        return coin, None

# --- 3. Parallel API calls (fast + safe) ---
dfs = {}
start = time.time()
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(fetch_coin, coin, 365) for coin in coin_ids if coin not in excluded]
    for future in as_completed(futures):
        coin, data = future.result()
        if data is not None:
            dfs[coin] = data
        time.sleep(1.2)  # respect API rate limits

# --- 4. Combine and save ---
price_df = pd.concat(dfs, axis=1).dropna()
price_df.to_csv("crypto_prices.csv")

end = time.time()
print(f"\n✅ Data for {len(dfs)} coins saved to 'crypto_prices.csv' in {round(end - start, 1)} seconds.")


Fetching daily prices for 21 major non-stable cryptocurrencies:
['bitcoin', 'ethereum', 'ripple', 'bnb', 'solana', 'tron', 'dogecoin', 'cardano', 'hyperliquid', 'chainlink', 'bitcoin-cash', 'stellar', 'unus-sed-leo', 'sui', 'hedera', 'avalanche-2', 'litecoin', 'monero', 'zcash', 'shiba-inu', 'the-open-network']
✅ monero fetched successfully.
⚠️ Error fetching the-open-network: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/the-open-network/market_chart?vs_currency=usd&days=365
✅ avalanche-2 fetched successfully.
✅ litecoin fetched successfully.
✅ zcash fetched successfully.
✅ shiba-inu fetched successfully.
✅ bittensor fetched successfully.
✅ crypto-com-chain fetched successfully.
✅ ethena-staked-usde fetched successfully.
✅ usdt0 fetched successfully.
⚠️ Error fetching ripple: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/ripple/market_chart?vs_currency=usd&days=365
✅ ethereum fetched successfully.
⚠️ Error fet

In [4]:
import pandas as pd
import numpy as np
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.expected_returns import mean_historical_return

# --- 1. Load cached price data ---
price_df = pd.read_csv("/crypto_prices.csv", index_col=0, parse_dates=True)
print("\nPrice Table (last 5 rows):")
print(price_df.tail())

# --- 2. Compute expected returns & covariance matrix ---
mu = mean_historical_return(price_df)   # expected annual returns
S = CovarianceShrinkage(price_df).ledoit_wolf()  # robust covariance estimator

# --- 3. Initialise the Efficient Frontier (no constraints) ---
ef = EfficientFrontier(mu, S)

# --- 4. Compute Portfolios ---
# (a) Minimum Volatility Portfolio
ef_min = ef.deepcopy()
w_min = ef_min.min_volatility()
cleaned_min = ef_min.clean_weights()

# (b) Maximum Sharpe Ratio Portfolio
ef_sharpe = ef.deepcopy()
w_sharpe = ef_sharpe.max_sharpe()
cleaned_sharpe = ef_sharpe.clean_weights()

# --- 5. Display Results ---
print("\n--- Minimum Risk Portfolio ---")
for k, v in cleaned_min.items():
    if v > 0.001:
        print(f"{k}: {v:.2%}")

print("\n--- Maximum Sharpe Portfolio ---")
for k, v in cleaned_sharpe.items():
    if v > 0.001:
        print(f"{k}: {v:.2%}")



Price Table (last 5 rows):
               ethereum        bitcoin      tron   cardano  dogecoin  \
date                                                                   
2025-10-30  3897.359268  110046.669258  0.296112  0.639642  0.192355   
2025-10-31  3802.295365  108240.765287  0.292382  0.600502  0.182665   
2025-11-01  3847.298177  109573.905556  0.296202  0.609137  0.186458   
2025-11-02  3872.211896  110014.135568  0.297450  0.612463  0.187270   
2025-11-03  3910.094769  110650.209282  0.298180  0.608449  0.186381   

            hyperliquid  chainlink   stellar       sui    litecoin  \
date                                                                 
2025-10-30    47.935208  18.082754  0.315766  2.510382   98.554944   
2025-10-31    45.473036  16.790154  0.298441  2.288553   93.329113   
2025-11-01    43.648900  17.238135  0.305038  2.364425   95.566795   
2025-11-02    43.250700  17.105318  0.304972  2.378123  101.375500   
2025-11-03    42.440303  17.564705  0.304949  2